**Oil Price Pass-Through to U.S. Inflation: Evidence from Diesel and WTI**

In [3]:
import requests
import pandas as pd
import numpy as np
from fredapi import Fred

In [4]:
### EIA data api
API_KEY = "5SD8YvpK2hQOMWzUrs9KbLg7cLFp8bwCY9F2ZoOv"
url = "https://api.eia.gov/v2/petroleum/pri/spt/data/"

#### the EIA have a helper to create this params, so i guided from there, 
# https://www.eia.gov/opendata/browser/petroleum/pri/spt?frequency=monthly&data=value;&sortColumn=period;&sortDirection=desc;
#### although it basically helped me i used ai to finish tunning it. 

params = {
    "api_key": API_KEY, 
    "frequency": "monthly", # frequency
    "data[0]": "value",  # [0] means it's the first item in an array of column names, and "value" is the column being requested
    "facets[product][]": ["EPCWTI", "EPD2DXL0"], # filters results to only these two series id, that are the ones the i want
    "sort[0][column]": "period", # date column 
    "sort[0][direction]": "asc", # ascending order (older to newer)
    "offset": 0, # starts from where the data is available
    "length": 5000 
}

response = requests.get(url, params=params)
data = response.json() 
# data

data = data.get("response", {}).get("data", []) #is how i parse the response you get back from the data
df = pd.DataFrame(data) # creating the data frame
df
## for the diesel code there are two duoarea so i just wanted to stick with one therefore delete the other one
## the other duoarea wa just of the New york surroundings. 
df = df[df["duoarea"] != "Y35NY"] ## delted this rows 

## i filtered because my columns of interest where this 3
df = df[["period", "product-name", "value"]]
df

,period,product-name,value
0,1986-01,WTI Crude Oil,22.93
1,1986-02,WTI Crude Oil,15.46
2,1986-03,WTI Crude Oil,12.61
3,1986-04,WTI Crude Oil,12.84
4,1986-05,WTI Crude Oil,15.38
...,...,...,...
954,2026-02,WTI Crude Oil,64.51
956,2026-03,No 2 Diesel Low Sulfur (0-15 ppm),3.832
957,2026-03,WTI Crude Oil,91.38
959,2026-04,No 2 Diesel Low Sulfur (0-15 ppm),3.915


In [5]:
### Dataframe in wider format
df_wide = df.pivot(
    index="period",
    columns="product-name",
    values="value").reset_index() # i do the reset because if not the period column becomes the index column and
# i want to be able to select the period column 

# df_wide
eia_df = df_wide.rename(columns={
    "period" : "date",
    "WTI Crude Oil": "wti",
    "No 2 Diesel Low Sulfur (0-15 ppm)": "diesel"})

### did this to do the merging forward
eia_df["date"] = pd.to_datetime(eia_df["date"])
eia_df


product-name,date,diesel,wti
0,1986-01-01,NaN,22.93
1,1986-02-01,NaN,15.46
2,1986-03-01,NaN,12.61
3,1986-04-01,NaN,12.84
4,1986-05-01,NaN,15.38
...,...,...,...
479,2025-12-01,2.072,57.97
480,2026-01-01,2.111,60.04
481,2026-02-01,2.302,64.51
482,2026-03-01,3.832,91.38


In [6]:
#### data de la FRED 
fred = Fred(api_key='b42654725fdba4223aab711d21e3fd8d')

cpi = fred.get_series("CPIAUCSL")
core_cpi = fred.get_series("CPILFESL")

cpi_df = cpi.reset_index()
cpi_df.columns = ["date", "cpi"]

core_df = core_cpi.reset_index()
core_df.columns = ["date", "core_cpi"]

fred_df = cpi_df.merge(core_df, on="date")
fred_df

,date,cpi,core_cpi
0,1957-01-01,27.670,28.500
1,1957-02-01,27.800,28.600
2,1957-03-01,27.860,28.700
3,1957-04-01,27.930,28.800
4,1957-05-01,28.000,28.800
...,...,...,...
827,2025-12-01,326.031,331.814
828,2026-01-01,326.588,332.793
829,2026-02-01,327.460,333.512
830,2026-03-01,330.293,334.165


In [ ]:
### data cleaning and transformation
final_df = fred_df.merge(eia_df, on="date")
# final_df

cols = ['wti', 'diesel' , 'cpi', 'core_cpi']
## the EIA data was not numeric so i decided to make sure every number in the data was numeric to apply this next line. 
final_df[cols] = final_df[cols].astype(int)
final_df

# final_df.to_csv("df_data.csv", index=False)